# Google Local New York Dataset Exploration

## Project
Context-Aware Activity Recommendation System

## Purpose

This notebook explores the Google Local New York dataset to determine how it can be used for our activity recommendation system.

Our initial goals are to:

1. Examine the review and business metadata.
2. Identify categories that represent activities.
3. Filter out businesses that are not relevant to activity recommendation.
4. Examine user interaction histories.
5. Determine whether users have enough activity interactions for personalized recommendation.
6. Create a smaller dataset for later recommendation experiments.

## Loading the Dataset

The project uses the Google Local New York dataset.

Two files are used:

- **Review data:** Contains user interactions, ratings, review text, timestamps, and place identifiers.
- **Metadata:** Contains information about each place, including its name, categories, location, average rating, and number of reviews.

The datasets can be connected using the `gmap_id` field.

In [18]:
import pandas as pd
from pathlib import Path

### Dataset Setup

Before running this notebook, download the Google Local New York review and metadata files and place them in:

`data/raw/`

Expected project structure:

Activity-Recommendation-System/
- data/
  - raw/
    - meta-New_York.json.gz
    - review-New_York_10 2.json
  - processed/
- notebooks/
  - 01_data_exploration.ipynb

In [26]:
PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent
elif (PROJECT_DIR / "Activity-Recommendation-System").exists():
    PROJECT_DIR = PROJECT_DIR / "Activity-Recommendation-System"

DATA_DIR = PROJECT_DIR / "data" / "raw"

print("Project directory:", PROJECT_DIR)
print("Data directory:", DATA_DIR)
print("Data directory exists:", DATA_DIR.exists())

Project directory: /Users/ogechikaesedebe/Downloads/Activity-Recommendation-System
Data directory: /Users/ogechikaesedebe/Downloads/Activity-Recommendation-System/data/raw
Data directory exists: True


In [27]:
if len(metadata_files) != 1:
    raise FileNotFoundError(
        "Expected one metadata file in data/raw/. "
        "Please check the dataset setup instructions above."
    )

if len(review_files) != 1:
    raise FileNotFoundError(
        "Expected one review file in data/raw/. "
        "Please check the dataset setup instructions above."
    )

metadata_path = metadata_files[0]
review_path = review_files[0]

print("Metadata:", metadata_path.name)
print("Reviews:", review_path.name)

Metadata: meta-New_York.json.gz
Reviews: review-New_York_10 2.json


## Load Place Metadata

In [28]:
metadata = pd.read_json(
    metadata_path,
    lines=True,
    compression="gzip"
)

print("Metadata shape:", metadata.shape)
metadata.head()

Metadata shape: (272189, 15)


,name,address,gmap_id,description,latitude,longitude,category,avg_rating,num_of_reviews,price,hours,MISC,state,relative_results,url
0,A-Top Insurance,"A-Top Insurance, 1009 Brighton Beach Ave, Broo...",0x89c24469c758686b:0x641f5b84cb9bedfa,None,40.578254,-73.959127,"[Insurance broker, Insurance agency]",2.0,4,None,"[[Thursday, 10AM–6PM], [Friday, 10AM–6PM], [Sa...",None,Open ⋅ Closes 6PM,"[0x89c24449907718fb:0x31b554a0983f621d, 0x4065...",https://www.google.com/maps/place//data=!4m2!3...
1,T-Mobile,"T-Mobile, 3923 103rd St, Queens, NY 11368",0x89c25fc9494dce47:0x6d63c807b59a55,None,40.750146,-73.862536,"[Cell phone store, Electronic parts supplier, ...",3.5,95,$$,"[[Thursday, 10AM–8PM], [Friday, 10AM–8PM], [Sa...","{'Service options': ['In-store shopping', 'Del...",Open ⋅ Closes 8PM,"[0x89c25fc7a91c609f:0xb103d6a261373fd, 0x89c25...",https://www.google.com/maps/place//data=!4m2!3...
2,Ace Hardware,"Ace Hardware, 130 4th Ave, New York, NY 10003",0x89c259992463b6e1:0x4a601a43752541dd,"Chain retailer stocking household tools, suppl...",40.733182,-73.990201,"[Hardware store, Building materials store, Ele...",3.4,8,None,"[[Thursday, 8AM–6:30PM], [Friday, 8AM–6:30PM],...",None,Permanently closed,"[0x89c2598340fb6509:0xa996f3db49ba6e78, 0x89c2...",https://www.google.com/maps/place//data=!4m2!3...
3,"Agricultural Consulting Services, Inc.","Agricultural Consulting Services, Inc., 370 Su...",0x89d14bf74320aea5:0x720261cd8960f8cd,None,43.054743,-77.649136,[Corporate office],3.0,1,None,"[[Thursday, 9AM–5PM], [Friday, 9AM–5PM], [Satu...",None,Open ⋅ Closes 5PM,None,https://www.google.com/maps/place//data=!4m2!3...
4,Long Point,"Long Point, Ledyard, NY 13026",0x89d0bd8d2df8d10f:0x77a8f349f40492fa,None,42.715626,-76.710227,[Peninsula],5.0,1,None,None,None,None,None,"https://www.google.com/maps/place/Ledyard,+NY+..."


## Load Review Sample (100,000)

In [29]:
reviews_sample = pd.read_json(
    review_path,
    lines=True,
    nrows=100000
)

print("Review sample shape:", reviews_sample.shape)
reviews_sample.head()

Review sample shape: (100000, 8)


,user_id,name,time,rating,text,pics,resp,gmap_id
0,1.137221e+20,Alvin Martinez,1603494795361,5,I'm late to posting this but this store especi...,[{'url': ['https://lh5.googleusercontent.com/p...,None,0x89c25fc9494dce47:0x6d63c807b59a55
1,1.072934e+20,Johnnie Jackson,1620157037403,1,Very dissatisfied I did not get my phone the p...,None,"{'time': 1620268360920, 'text': 'We pride ours...",0x89c25fc9494dce47:0x6d63c807b59a55
2,1.003786e+20,Manie Blazer,1597431662039,5,Excellent very well done with professional car...,None,None,0x89c25fc9494dce47:0x6d63c807b59a55
3,1.149982e+20,Fashion Fiinds,1543773862044,5,Basing my review strictly on the service I rec...,None,"{'time': 1543855317372, 'text': 'Thanks for th...",0x89c25fc9494dce47:0x6d63c807b59a55
4,1.171782e+20,Andres Rieloff,1597279097718,1,Bad! Disorganized. I'm being totally honest. I...,None,None,0x89c25fc9494dce47:0x6d63c807b59a55


## Dataset Overview

Before filtering the dataset for activities, we examine the structure and key fields available in the metadata and review data.

In [30]:
print("Metadata columns:")
print(metadata.columns.tolist())

print("\nReview columns:")
print(reviews_sample.columns.tolist())

Metadata columns:
['name', 'address', 'gmap_id', 'description', 'latitude', 'longitude', 'category', 'avg_rating', 'num_of_reviews', 'price', 'hours', 'MISC', 'state', 'relative_results', 'url']

Review columns:
['user_id', 'name', 'time', 'rating', 'text', 'pics', 'resp', 'gmap_id']


### Missing Values

We examine missing values to determine which fields contain enough information to be useful in later stages of the project.

In [33]:
metadata_missing = pd.DataFrame({
    "missing_count": metadata.isnull().sum(),
    "missing_percent": metadata.isnull().mean() * 100
}).sort_values("missing_percent", ascending=False)

metadata_missing.round(2)



review_missing = pd.DataFrame({
    "missing_count": reviews_sample.isnull().sum(),
    "missing_percent": reviews_sample.isnull().mean() * 100
}).sort_values("missing_percent", ascending=False)

review_missing.round(2)



print("Metadata missing values:")
display(metadata_missing)

print("\nReview sample missing values:")
display(review_missing)

Metadata missing values:


,missing_count,missing_percent
description,217696,79.979720
price,213559,78.459820
state,77249,28.380647
hours,58077,21.337012
MISC,53773,19.755758
relative_results,19928,7.321383
address,2732,1.003714
category,1013,0.372168
name,2,0.000735
gmap_id,0,0.000000



Review sample missing values:


,missing_count,missing_percent
pics,96968,96.968
resp,89117,89.117
text,41428,41.428
user_id,0,0.000
rating,0,0.000
time,0,0.000
name,0,0.000
gmap_id,0,0.000


### Notes
-**Note:** Missing-value statistics for the metadata are based on the complete New York metadata dataset. Missing-value statistics for the review data are currently based on the first 100,000 reviews loaded for initial exploration. This review subset is used to examine the structure of the data and has not been selected as the final dataset for modeling.